## Patch features

This module takes a chip and computes patch-level metrics, i.e. sar statistics, era5 bands, amsr2 bands, etc.

In [ ]:
# def compute_patch_features(chip):
#     """
#     Given a prepared chip from iter_chips(), compute patch-level features
#     for all 1024 patches in the 32x32 grid.

#     Returns list of 1024 dicts, one per patch, in row-major order.
#     """
#     valid    = chip["valid_mask"]           # (256, 256) bool
#     sar      = chip["bands"]           # dict of (256, 256) arrays
#     distance = chip["distance"]        # (256, 256)
#     ancillary = chip["bands"]          # resampled AMSR2 + ERA5 already in here

#     hh = sar["nersc_sar_primary"]      # (256, 256)
#     hv = sar["nersc_sar_secondary"]    # (256, 256)

#     # Load incidence angle — resampled to SAR resolution by B2.1
#     # Assuming it was added to chip["bands"] or chip dict by loader
#     ia = chip.get("incidence_angle_map")   # (256, 256) resampled

#     patch_records = []

#     for pi in range(GRID_SIZE):
#         for pj in range(GRID_SIZE):

#             # ── Pixel slice for this patch ────────────────────────────────────
#             r0, r1 = pi * PATCH_SIZE, (pi + 1) * PATCH_SIZE
#             c0, c1 = pj * PATCH_SIZE, (pj + 1) * PATCH_SIZE

#             valid_patch = valid[r0:r1, c0:c1]          # (8, 8) bool
#             hh_patch    = hh[r0:r1, c0:c1]             # (8, 8)
#             hv_patch    = hv[r0:r1, c0:c1]             # (8, 8)

#             # ── Valid fraction ────────────────────────────────────────────────
#             # Count over boolean mask BEFORE substitution (64 pixels total)
#             valid_frac = valid_patch.sum() / 64.0

#             # ── SAR statistics (valid pixels only) ───────────────────────────
#             hh_valid = hh_patch[valid_patch]
#             hv_valid = hv_patch[valid_patch]

#             if len(hh_valid) > 0:
#                 hh_mean = float(hh_valid.mean())
#                 hh_std  = float(hh_valid.std())
#                 hv_mean = float(hv_valid.mean())
#                 hv_std  = float(hv_valid.std())
#                 # HV/HH ratio in linear space (values are in dB so subtract)
#                 hv_hh_ratio = float((hv_valid - hh_valid).mean())
#             else:
#                 hh_mean = hh_std = hv_mean = hv_std = hv_hh_ratio = np.nan

#             # ── Patch centroid (pixel coords) ─────────────────────────────────
#             rc = r0 + PATCH_SIZE // 2    # row centre
#             cc = c0 + PATCH_SIZE // 2    # col centre

#             # ── Ancillary centroid samples ────────────────────────────────────
#             # AMSR2 and ERA5 already resampled to SAR resolution by B2.1
#             # Just sample at centroid — no averaging needed at this scale
#             amsr2_sample = {
#                 var: float(ancillary[var][rc, cc])
#                 for var in AMSR2_BANDS
#             }

#             era5_sample = {
#                 var: float(ancillary[var][rc, cc])
#                 for var in ERA5_BANDS
#             }

#             # ── Distance to land at centroid ──────────────────────────────────
#             dist = float(distance[rc, cc])

#             # ── Incidence angle mean ──────────────────────────────────────────
#             if ia is not None:
#                 ia_patch = ia[r0:r1, c0:c1]
#                 ia_mean  = float(ia_patch[valid_patch].mean()) if valid_patch.any() else np.nan
#             else:
#                 ia_mean = np.nan

#             # ── Assemble patch record ─────────────────────────────────────────
#             record = {
#                 "chip_idx":    chip["chip_idx"],
#                 "patch_i":     pi,
#                 "patch_j":     pj,
#                 "valid_frac":  valid_frac,
#                 "hh_mean":     hh_mean,
#                 "hv_mean":     hv_mean,
#                 "hh_std":      hh_std,
#                 "hv_std":      hv_std,
#                 "hv_hh_ratio": hv_hh_ratio,
#                 "ia_mean":     ia_mean,
#                 "distance":    dist,
#                 **amsr2_sample,
#                 **era5_sample,
#             }

#             patch_records.append(record)

#     return patch_records

In [ ]:
# # Manually verify patch [2, 3] as the spec requires
# pi, pj = 2, 3
# r0, r1 = pi*8, (pi+1)*8
# c0, c1 = pj*8, (pj+1)*8

# manual_valid_frac = chip["valid_mask"][r0:r1, c0:c1].sum() / 64.0
# module_valid_frac = patches[pi * GRID_SIZE + pj]["valid_frac"]

# print(f"Manual valid_frac:  {manual_valid_frac:.4f}")
# print(f"Module valid_frac:  {module_valid_frac:.4f}")
# print(f"Match: {np.isclose(manual_valid_frac, module_valid_frac)}")

Manual valid_frac:  0.0000
Module valid_frac:  0.0000
Match: True


In [ ]:
# chip_rows  = []
# patch_rows = []

# for chip in yield_chips():
#     # B2.2 — embeddings
#     patch_tokens, class_token = encode_chip(chip)

#     # B2.3 — patch features
#     patches = compute_patch_features(chip)

#     # Add embedding to each patch record
#     for pi in range(GRID_SIZE):
#         for pj in range(GRID_SIZE):
#             idx = pi * GRID_SIZE + pj
#             patches[idx]["embedding"] = patch_tokens[pi, pj, :]

#     # Chip table row
#     chip_rows.append({
#         "chip_idx":  chip["chip_idx"],
#         "lat":       chip["lat"],
#         "lon":       chip["lon"],
#         "embedding": class_token,
#     })

#     # Patch table rows
#     patch_rows.extend(patches)

# print(f"Chips: {len(chip_rows)}")
# print(f"Patches: {len(patch_rows)}")   # should be chips * 1024

KeyError: 'bands'